# Advanced NumPy — Interview Preparation Guide

**Target audience:** Data science/ML candidates preparing for technical interviews  
**Level:** Intermediate → Advanced  
**Prerequisites:** NumPy basics (arrays, slicing, basic operations)

---

## Topics Covered

1. Views vs Copies  
2. Broadcasting Rules In Depth  
3. Memory Layout: C-order vs F-order  
4. Advanced Indexing  
5. Vectorization vs Loops  
6. `np.einsum`  
7. Structured Arrays and Record Arrays  
8. Linear Algebra Deep Dive  
9. Random Number Generation  
10. Performance Tricks  
11. `np.apply_along_axis` vs Vectorized  
12. Masked Arrays  
13. Common Interview Traps  

---

In [1]:
import numpy as np
import time
import sys

print(f'NumPy version: {np.__version__}')
print(f'Python version: {sys.version}')

NumPy version: 2.4.6
Python version: 3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]


---
## 1. Views vs Copies

One of the most dangerous and frequently tested NumPy concepts. A **view** shares memory with the original array — modifying it modifies the original. A **copy** is independent.

### When NumPy returns a VIEW:
- Basic slicing: `a[2:5]`, `a[::2]`, `a[:,0]`
- Reshaping (when possible): `a.reshape(...)` — *usually* a view
- Transposing: `a.T`

### When NumPy returns a COPY:
- Fancy (integer array) indexing: `a[[0,1,2]]`
- Boolean indexing: `a[a > 0]`
- `np.copy()` or `.copy()`
- Operations that create new dtypes

In [2]:
# --- Basic slicing returns a VIEW ---
a = np.array([1, 2, 3, 4, 5])
b = a[1:4]   # view

print('Before modification:')
print('a:', a)
print('b:', b)

b[0] = 99   # modifies a too!

print('\nAfter b[0] = 99:')
print('a:', a)  # a is changed!
print('b:', b)

# Check with .base attribute
# If b.base is a, then b is a view of a
print('\nb.base is a:', b.base is a)   # True -> b is a view

Before modification:
a: [1 2 3 4 5]
b: [2 3 4]

After b[0] = 99:
a: [ 1 99  3  4  5]
b: [99  3  4]

b.base is a: True


In [3]:
# --- Fancy indexing returns a COPY ---
a = np.array([1, 2, 3, 4, 5])
c = a[[1, 2, 3]]   # fancy indexing -> COPY

print('c.base:', c.base)  # None -> c is NOT a view

c[0] = 99
print('a after modifying c:', a)  # a unchanged

c.base: None
a after modifying c: [1 2 3 4 5]


In [4]:
# --- Boolean indexing returns a COPY ---
a = np.array([1, -2, 3, -4, 5])
d = a[a > 0]

print('d.base:', d.base)  # None -> copy

d[0] = 999
print('a after modifying d:', a)  # a unchanged

d.base: None
a after modifying d: [ 1 -2  3 -4  5]


In [5]:
# --- Reshape: usually a view ---
a = np.arange(12)
b = a.reshape(3, 4)

print('b.base is a:', b.base is a)  # True

b[0, 0] = 999
print('a[0]:', a[0])  # 999 - shared memory!

b.base is a: True
a[0]: 999


In [6]:
# --- How to check if two arrays share memory ---
a = np.arange(10)
b = a[::2]
c = a.copy()

print('np.shares_memory(a, b):', np.shares_memory(a, b))   # True
print('np.shares_memory(a, c):', np.shares_memory(a, c))   # False

# Safe pattern: always copy when in doubt
safe = b.copy()
print('safe.base:', safe.base)   # None

np.shares_memory(a, b): True
np.shares_memory(a, c): False
safe.base: None


> ### INTERVIEW QUESTION
> **Q: What is the difference between a view and a copy in NumPy? When does NumPy return each?**
>
> **A:** A view shares the same memory buffer as the original array — any modification to the view modifies the original. A copy allocates new memory and is fully independent. NumPy returns **views** for basic/slice indexing and most reshape/transpose operations. It returns **copies** for fancy (integer array) indexing, boolean indexing, and explicit `.copy()` calls. You can verify with `.base` attribute (None means owns its data) or `np.shares_memory()`.
>
> ### WHY THIS MATTERS for ML/DS
> Silent bugs: normalizing a slice of a feature matrix in-place can corrupt your original data. Always use `.copy()` when passing array slices to functions that mutate inputs.

---
## 2. Broadcasting Rules In Depth

Broadcasting allows NumPy to perform operations on arrays with different shapes without making copies. Mastering it eliminates most loops.

### The 3 Rules (applied right-to-left on shape tuples):

```
Rule 1: If arrays have different number of dimensions,
        the shape of the smaller array is padded with 1s on the LEFT.

Rule 2: Arrays with size 1 along a dimension act as if they have
        the size of the array with the largest size in that dimension.

Rule 3: If arrays disagree in size in any dimension AND neither is 1,
        a ValueError is raised.
```

### ASCII Diagram:

```
A shape:  (3, 4)      ->  (3, 4)
B shape:     (4,)     ->  (1, 4)   [Rule 1: pad left]
                      ->  (3, 4)   [Rule 2: stretch dim 0]
Result:   (3, 4)      OK

A shape:  (3, 1)
B shape:  (1, 4)
Result:   (3, 4)      OK Both stretched

A shape:  (3, 4)
B shape:  (3, 3)
Result:   ERROR        dim 1: 4 vs 3, neither is 1
```

In [7]:
# Rule 1 + 2: 2D + 1D
A = np.ones((3, 4))
b = np.array([1, 2, 3, 4])   # shape (4,) -> padded to (1,4) -> stretched to (3,4)

result = A + b
print('A shape:', A.shape)
print('b shape:', b.shape)
print('result shape:', result.shape)
print(result)

A shape: (3, 4)
b shape: (4,)
result shape: (3, 4)
[[2. 3. 4. 5.]
 [2. 3. 4. 5.]
 [2. 3. 4. 5.]]


In [8]:
# Both arrays need stretching -- classic outer operation pattern
row = np.array([0, 1, 2, 3])          # shape (4,)
col = np.array([0, 10, 20]).reshape(3, 1)  # shape (3, 1)

outer = row + col
print('row shape:', row.shape)
print('col shape:', col.shape)
print('result shape:', outer.shape)
print(outer)

row shape: (4,)
col shape: (3, 1)
result shape: (3, 4)
[[ 0  1  2  3]
 [10 11 12 13]
 [20 21 22 23]]


In [9]:
# Tricky: column normalization of a 2D matrix
data = np.array([[1., 2., 3.],
                 [4., 5., 6.],
                 [7., 8., 9.]])

col_means = data.mean(axis=0)    # shape (3,)
col_stds  = data.std(axis=0)     # shape (3,)

normalized = (data - col_means) / col_stds   # broadcasting in action
print('col_means shape:', col_means.shape)
print('normalized:\n', normalized)

col_means shape: (3,)
normalized:
 [[-1.22474487 -1.22474487 -1.22474487]
 [ 0.          0.          0.        ]
 [ 1.22474487  1.22474487  1.22474487]]


In [10]:
# TRICKY EDGE CASE: shape (n,) vs (n,1) -- a common interview trap
a = np.array([1, 2, 3])    # shape (3,)
b = np.array([[1],[2],[3]])  # shape (3,1)

print('a shape:', a.shape)
print('b shape:', b.shape)
print('a + b shape:', (a + b).shape)   # (3,3) !! NOT (3,)
print(a + b)  # outer-product-like addition

a shape: (3,)
b shape: (3, 1)
a + b shape: (3, 3)
[[2 3 4]
 [3 4 5]
 [4 5 6]]


In [11]:
# Broadcasting error example
try:
    x = np.ones((3, 4))
    y = np.ones((3, 3))
    z = x + y
except ValueError as e:
    print('Error:', e)

Error: operands could not be broadcast together with shapes (3,4) (3,3) 


> ### INTERVIEW QUESTION
> **Q: Explain NumPy broadcasting rules. What shape results from adding arrays of shape (5, 1, 4) and (3, 4)?**
>
> **A:** The 3 rules: (1) pad the shorter shape with 1s on the left, (2) dimensions of size 1 are stretched to match, (3) incompatible sizes raise an error. For (5,1,4) + (3,4): first pad (3,4) to (1,3,4). Check dim by dim right-to-left: 4==4 OK, 1 vs 3 -> stretch -> 3 OK, 5 vs 1 -> stretch -> 5 OK. Result: **(5, 3, 4)**.
>
> ### WHY THIS MATTERS for ML/DS
> Matrix normalization, computing pairwise distances, adding bias vectors -- broadcasting is the core mechanism behind vectorized ML code. Getting shapes wrong silently produces wrong-shaped results.

---
## 3. Memory Layout: C-order vs F-order

Understanding memory layout is essential for writing cache-friendly code and interfacing with external libraries.

### C-order (Row-major) -- NumPy default
- Row elements are contiguous in memory
- Last index changes fastest
- Same as C, C++, Python

### F-order (Column-major) -- Fortran order
- Column elements are contiguous in memory
- First index changes fastest
- Same as Fortran, MATLAB, R

```
Matrix:  [[1, 2, 3],
          [4, 5, 6]]

C-order memory:   1 2 3 4 5 6   (row by row)
F-order memory:   1 4 2 5 3 6   (column by column)
```

### Strides
Strides tell NumPy how many **bytes** to step in each dimension.

In [12]:
# C-order (default) vs F-order
c_arr = np.array([[1,2,3],[4,5,6]], order='C')
f_arr = np.array([[1,2,3],[4,5,6]], order='F')

print('C-order strides:', c_arr.strides)   # (24, 8) -> 8 bytes per int64, 3 elements per row
print('F-order strides:', f_arr.strides)   # (8, 16) -> 8 bytes per element, 2 elements per col

print('C contiguous:', c_arr.flags['C_CONTIGUOUS'])
print('F contiguous:', f_arr.flags['F_CONTIGUOUS'])

C-order strides: (24, 8)
F-order strides: (8, 16)
C contiguous: True
F contiguous: True


In [13]:
# Strides explain views
a = np.arange(20).reshape(4, 5)
print('Original strides:', a.strides)    # (40, 8) -- 5 int64s per row

# Every other column -- NOT contiguous
b = a[:, ::2]
print('Sliced strides:', b.strides)      # (40, 16) -- step 2 cols
print('C contiguous:', b.flags['C_CONTIGUOUS'])   # False!
print('Is view:', b.base is not None)   # True -- it is a view

Original strides: (40, 8)
Sliced strides: (40, 16)
C contiguous: False
Is view: True


In [14]:
# Performance: row-wise vs column-wise access in C-order array
size = 1000
arr = np.random.randn(size, size)

# Row-wise sum (cache-friendly for C-order)
t0 = time.perf_counter()
for _ in range(100):
    _ = arr.sum(axis=1)
t_row = time.perf_counter() - t0

# Column-wise sum (cache-unfriendly for C-order)
t0 = time.perf_counter()
for _ in range(100):
    _ = arr.sum(axis=0)
t_col = time.perf_counter() - t0

print(f'Row-wise sum (axis=1):    {t_row*1000:.2f} ms')
print(f'Column-wise sum (axis=0): {t_col*1000:.2f} ms')
print(f'Ratio: {t_col/t_row:.2f}x')

Row-wise sum (axis=1):    42.86 ms
Column-wise sum (axis=0): 51.88 ms
Ratio: 1.21x


In [15]:
# ascontiguousarray -- useful before passing to C extensions
a = np.arange(20).reshape(4, 5)
b = a[:, ::2]  # non-contiguous
print('b contiguous?', b.flags['C_CONTIGUOUS'])

b_cont = np.ascontiguousarray(b)
print('b_cont contiguous?', b_cont.flags['C_CONTIGUOUS'])
print('b_cont strides:', b_cont.strides)

b contiguous? False
b_cont contiguous? True
b_cont strides: (24, 8)


> ### INTERVIEW QUESTION
> **Q: What are array strides in NumPy, and why does memory layout matter for performance?**
>
> **A:** Strides are a tuple indicating the number of bytes to step in each dimension. For a C-order (row-major) array of shape (M, N) with float64, strides are (N*8, 8). Memory layout matters because CPUs load data in cache lines -- accessing memory sequentially (cache-friendly) is much faster than jumping around. C-order arrays are fast for row-wise operations (axis=1); F-order for column-wise. Non-contiguous arrays can be 2-5x slower for certain operations.
>
> ### WHY THIS MATTERS for ML/DS
> NumPy defaults to C-order; BLAS/LAPACK (used by `np.dot`, `linalg`) may require or prefer specific layouts. Passing non-contiguous arrays to scipy or C extensions can silently trigger costly copies.

---
## 4. Advanced Indexing

NumPy has three indexing modes: basic (slices), integer array (fancy), and boolean. Each behaves differently.

### Integer Array Indexing (Fancy Indexing)
- Always returns a **copy**
- Can select arbitrary elements, rows, columns
- Can duplicate elements

### Boolean Indexing
- Always returns a **copy**
- Mask must match shape along indexed dimension
- `np.where`, `np.select` are vectorized conditionals

In [16]:
# --- Integer array indexing ---
a = np.array([10, 20, 30, 40, 50])

# Select elements by index
idx = [0, 2, 4, 2]   # can repeat!
print('Fancy index:', a[idx])    # [10, 30, 50, 30]

# 2D: select specific (row, col) pairs
M = np.arange(16).reshape(4, 4)
rows = [0, 1, 2]
cols = [1, 2, 3]
print('M:\n', M)
print('M[rows, cols]:', M[rows, cols])   # M[0,1], M[1,2], M[2,3]

Fancy index:

 [10 30 50 30]
M:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]
M[rows, cols]: [ 1  6 11]


In [17]:
# --- Select entire rows (common in ML: gather operation) ---
M = np.arange(20).reshape(5, 4)
row_indices = [0, 2, 4]   # gather specific rows
print('Selected rows:\n', M[row_indices])   # shape (3, 4)

Selected rows:
 [[ 0  1  2  3]
 [ 8  9 10 11]
 [16 17 18 19]]


In [18]:
# --- Boolean indexing ---
a = np.array([3, -1, 4, -1, 5, -9, 2, 6])

mask = a > 0
print('Mask:', mask)
print('Positive values:', a[mask])

# Modify in-place via boolean mask
a[a < 0] = 0
print('After zeroing negatives:', a)

Mask: [ True False  True False  True False  True  True]
Positive values: [3 4 5 2 6]
After zeroing negatives: [3 0 4 0 5 0 2 6]


In [19]:
# --- np.where: vectorized if-else ---
a = np.array([-3, -1, 0, 2, 5, -2, 7])

# np.where(condition, value_if_true, value_if_false)
result = np.where(a > 0, a, 0)   # ReLU!
print('ReLU:', result)

# np.where with single argument: returns indices where condition is True
indices = np.where(a > 0)
print('Positive indices:', indices)   # tuple of arrays

ReLU: [0 0 0 2 5 0 7]
Positive indices: (array([3, 4, 6]),)


In [20]:
# --- np.select: chained conditions (multiple branches) ---
a = np.array([-5, -1, 0, 1, 3, 7, 15])

conditions = [a < 0, a == 0, (a > 0) & (a < 5), a >= 5]
choices    = [-1,    0,      1,                   2]

result = np.select(conditions, choices, default=999)
print('np.select result:', result)

np.select result: [-1 -1  0  1  1  2  2]


In [21]:
# --- Fancy indexing pitfall: assignment via copy is lost ---
M = np.arange(12).reshape(3, 4)
print('Original:\n', M)

# This does NOT modify M -- M[[0,1]] is a copy:
# M[[0,1]][[0]] = 99   # WRONG

# Correct way to assign with fancy indexing:
M[[0, 1], [0, 2]] = 99
print('After M[[0,1],[0,2]] = 99:\n', M)

Original:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
After M[[0,1],[0,2]] = 99:
 [[99  1  2  3]
 [ 4  5 99  7]
 [ 8  9 10 11]]


In [22]:
# --- np.ix_: create open meshgrid for fancy indexing submatrix ---
M = np.arange(25).reshape(5, 5)
rows = [0, 2, 4]
cols = [1, 3]

# Select the 3x2 submatrix at rows [0,2,4] and cols [1,3]
submatrix = M[np.ix_(rows, cols)]
print('Submatrix shape:', submatrix.shape)   # (3, 2)
print(submatrix)

Submatrix shape: (3, 2)
[[ 1  3]
 [11 13]
 [21 23]]


> ### INTERVIEW QUESTION
> **Q: What is the difference between `a[0:3]` and `a[[0,1,2]]`? When would the distinction cause a bug?**
>
> **A:** `a[0:3]` is basic slicing -- returns a **view** sharing memory with `a`. `a[[0,1,2]]` is fancy indexing -- returns a **copy**. Bug scenario: `a[[0,1,2]] *= 2` silently does nothing to `a`, because the copy is modified and immediately discarded. With slicing, `a[0:3] *= 2` modifies `a` in-place.
>
> ### WHY THIS MATTERS for ML/DS
> Gather/scatter operations in deep learning map directly to fancy indexing. Embedding lookups (`W[token_ids]`) are fancy indexing on weight matrices.

---
## 5. Vectorization vs Loops

NumPy operations are implemented in C with SIMD instructions. Python loops add interpreter overhead (~100ns per iteration). The speedup from vectorization is typically 10-1000x.

### Key insight: `np.vectorize` is NOT actually fast
It's syntactic sugar that still calls your Python function element-by-element. It helps with broadcasting behavior, not performance.

In [23]:
# --- Pure Python loop vs NumPy vectorized ---
import time

N = 1_000_000
a = np.random.randn(N)
b = np.random.randn(N)

# Python loop
t0 = time.perf_counter()
c = [a[i] * b[i] for i in range(N)]
t_loop = time.perf_counter() - t0

# NumPy vectorized
t0 = time.perf_counter()
c = a * b
t_numpy = time.perf_counter() - t0

print(f'Python loop:  {t_loop*1000:.1f} ms')
print(f'NumPy:        {t_numpy*1000:.1f} ms')
print(f'Speedup:      {t_loop/t_numpy:.0f}x')

Python loop:  317.0 ms
NumPy:        18.4 ms
Speedup:      17x


In [24]:
# --- Replacing a for loop with NumPy operations ---
# Task: for each row in matrix, compute sum of squares

M = np.random.randn(10000, 100)

# Loop approach
t0 = time.perf_counter()
result_loop = np.array([np.sum(row**2) for row in M])
t_loop = time.perf_counter() - t0

# Vectorized
t0 = time.perf_counter()
result_vec = np.sum(M**2, axis=1)
t_vec = time.perf_counter() - t0

print(f'Loop:       {t_loop*1000:.2f} ms')
print(f'Vectorized: {t_vec*1000:.2f} ms')
print(f'Speedup:    {t_loop/t_vec:.1f}x')
print(f'Results match: {np.allclose(result_loop, result_vec)}')

Loop:       54.45 ms
Vectorized: 3.95 ms
Speedup:    13.8x
Results match: True


In [25]:
# --- np.vectorize: convenient but NOT fast ---
def my_func(x):
    if x > 0:
        return x ** 2
    else:
        return -x

# np.vectorize wraps the Python function -- still O(N) Python calls
vfunc = np.vectorize(my_func)

a = np.random.randn(100_000)

t0 = time.perf_counter()
r1 = vfunc(a)
t_vect = time.perf_counter() - t0

# True vectorized with np.where
t0 = time.perf_counter()
r2 = np.where(a > 0, a**2, -a)
t_numpy = time.perf_counter() - t0

print(f'np.vectorize: {t_vect*1000:.2f} ms')
print(f'np.where:     {t_numpy*1000:.2f} ms')
print(f'np.where speedup: {t_vect/t_numpy:.1f}x')
print(f'Results match: {np.allclose(r1, r2)}')

np.vectorize: 25.60 ms
np.where:     0.44 ms
np.where speedup: 57.5x
Results match: True


In [26]:
# --- Pattern: replacing conditional loops with vectorized ops ---
# Compute piecewise function:
# f(x) = x^2   if x > 1
#         x     if 0 <= x <= 1
#        -x^2   if x < 0

x = np.random.randn(1_000_000)

# Vectorized version (no loops):
result = np.select(
    [x > 1,    (x >= 0) & (x <= 1),  x < 0],
    [x**2,      x,                   -x**2]
)

print('Shape:', result.shape)
print('Sample:', result[:5])

Shape: (1000000,)
Sample: [ 3.50186160e-01 -7.88831344e-04  1.32674661e+00  1.01879679e+00
  5.80856679e-01]


> ### INTERVIEW QUESTION
> **Q: Is `np.vectorize` fast? When should you use it?**
>
> **A:** No. `np.vectorize` is **not** a performance optimization -- it still calls the Python function once per element, giving O(N) Python overhead. It's useful for (1) making a function broadcast correctly over arrays, and (2) applying functions with complex logic that can't easily be expressed with NumPy primitives. For performance, use `np.where`, `np.select`, or restructure the logic into NumPy array operations.
>
> ### WHY THIS MATTERS for ML/DS
> Training loops, preprocessing pipelines, and feature engineering are bottlenecks. Vectorizing these can reduce runtime from hours to minutes on large datasets.

---
## 6. `np.einsum` -- Einstein Summation

`np.einsum` is a generalized contraction notation. It can express matrix multiply, trace, transpose, outer products, batch operations, and more in a single readable notation.

### Notation: `'ij,jk->ik'`
- Each letter is an index
- Indices on the **right of `->` are kept** (appear in output)
- Indices **not on the right** are summed over (contracted)
- Repeated indices on **same tensor** -> trace/diagonal

This appears frequently in ML interviews for ML framework internals questions.

In [27]:
# --- Basic operations with einsum ---
A = np.random.randn(3, 4)
B = np.random.randn(4, 5)

# Matrix multiply: sum over j
C = np.einsum('ij,jk->ik', A, B)
print('Matrix multiply shape:', C.shape)   # (3,5)
print('Matches np.dot:', np.allclose(C, A @ B))

Matrix multiply shape: (3, 5)
Matches np.dot: True


In [28]:
# --- Trace: sum diagonal elements ---
A = np.arange(9).reshape(3, 3).astype(float)
trace_einsum = np.einsum('ii->', A)
trace_numpy  = np.trace(A)
print('Trace (einsum):', trace_einsum)
print('Trace (np.trace):', trace_numpy)

Trace (einsum): 12.0
Trace (np.trace): 12.0


In [29]:
# --- Element-wise multiply then sum (dot product) ---
a = np.array([1., 2., 3.])
b = np.array([4., 5., 6.])

dot_einsum = np.einsum('i,i->', a, b)
dot_numpy  = np.dot(a, b)
print('Dot (einsum):', dot_einsum)
print('Dot (np.dot):', dot_numpy)

Dot (einsum): 32.0
Dot (np.dot): 32.0


In [30]:
# --- Outer product ---
a = np.array([1., 2., 3.])
b = np.array([1., 2., 3., 4.])

outer_einsum = np.einsum('i,j->ij', a, b)
outer_numpy  = np.outer(a, b)
print('Outer product shape:', outer_einsum.shape)
print('Matches np.outer:', np.allclose(outer_einsum, outer_numpy))

Outer product shape: (3, 4)
Matches np.outer: True


In [31]:
# --- Batch matrix multiply (crucial in deep learning) ---
# Batch of 10 matrices: (batch, M, K) @ (batch, K, N) -> (batch, M, N)
batch_size, M, K, N = 10, 3, 4, 5
A = np.random.randn(batch_size, M, K)
B = np.random.randn(batch_size, K, N)

C_einsum = np.einsum('bij,bjk->bik', A, B)
C_matmul = A @ B   # @ also handles batches in NumPy

print('Batch matmul shape:', C_einsum.shape)   # (10, 3, 5)
print('Matches @ operator:', np.allclose(C_einsum, C_matmul))

Batch matmul shape: (10, 3, 5)
Matches @ operator: True


In [32]:
# --- Column sums, row sums, diagonal extraction ---
M = np.arange(1, 10).reshape(3, 3).astype(float)
print('Matrix:\n', M)

print('Col sums (einsum):', np.einsum('ij->j', M))
print('Col sums (np.sum):', M.sum(axis=0))

print('Row sums (einsum):', np.einsum('ij->i', M))

print('Diagonal (einsum):', np.einsum('ii->i', M))
print('Diagonal (np.diag):', np.diag(M))

Matrix:
 [[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]
Col sums (einsum): [12. 15. 18.]
Col sums (np.sum): [12. 15. 18.]
Row sums (einsum): [ 6. 15. 24.]
Diagonal (einsum): [1. 5. 9.]
Diagonal (np.diag): [1. 5. 9.]


In [33]:
# --- Quadratic form: x^T A x (appears in statistics/ML) ---
A = np.array([[2., 1.], [1., 3.]])
x = np.array([1., 2.])

quad_einsum = np.einsum('i,ij,j->', x, A, x)
quad_manual = x @ A @ x
print('Quadratic form (einsum):', quad_einsum)
print('Quadratic form (manual):', quad_manual)

Quadratic form (einsum): 18.0
Quadratic form (manual): 18.0


> ### INTERVIEW QUESTION
> **Q: What does `np.einsum('ijk,ikl->ijl', A, B)` compute?**
>
> **A:** It computes a batch matrix multiplication where the batch dimension is `i` and the matrix multiply contracts over `k`. For A of shape (I, J, K) and B of shape (I, K, L), it produces output of shape (I, J, L). Equivalent to `A @ B` when those shapes hold. The `k` index is summed over (contracted), while `i`, `j`, `l` are preserved.
>
> ### WHY THIS MATTERS for ML/DS
> Attention mechanisms in transformers use batch matrix multiplications. Understanding einsum notation helps read research paper equations and implement custom layers efficiently.

---
## 7. Structured Arrays and Record Arrays

Structured arrays allow each element to contain multiple fields of different types -- like a lightweight in-memory table without Pandas overhead.

In [34]:
# --- Define a structured array dtype ---
dtype = np.dtype([
    ('name',   'U10'),    # Unicode string, max 10 chars
    ('age',    np.int32),
    ('height', np.float64),
    ('active', np.bool_)
])

# Create structured array
people = np.array([
    ('Alice', 30, 1.65, True),
    ('Bob',   25, 1.80, False),
    ('Carol', 35, 1.70, True),
    ('Dave',  28, 1.75, True),
], dtype=dtype)

print('People array:\n', people)
print('\nDtype:', people.dtype)

People array:
 [('Alice', 30, 1.65,  True) ('Bob', 25, 1.8 , False)
 ('Carol', 35, 1.7 ,  True) ('Dave', 28, 1.75,  True)]

Dtype: [('name', '<U10'), ('age', '<i4'), ('height', '<f8'), ('active', '?')]


In [35]:
# --- Field access ---
print('All names:', people['name'])
print('All ages:', people['age'])

# Boolean selection on a field
adults = people[people['age'] > 27]
print('\nPeople older than 27:\n', adults)

# Sort by age
sorted_people = np.sort(people, order='age')
print('\nSorted by age:\n', sorted_people)

All names: ['Alice' 'Bob' 'Carol' 'Dave']
All ages: [30 25 35 28]

People older than 27:
 [('Alice', 30, 1.65,  True) ('Carol', 35, 1.7 ,  True)
 ('Dave', 28, 1.75,  True)]

Sorted by age:
 [('Bob', 25, 1.8 , False) ('Dave', 28, 1.75,  True)
 ('Alice', 30, 1.65,  True) ('Carol', 35, 1.7 ,  True)]


In [36]:
# --- Record arrays: field access via attribute notation ---
rec = people.view(np.recarray)
print('Names via attribute:', rec.name)
print('Heights:', rec.height)
print('Active members:', rec[rec.active])

Names via attribute: ['Alice' 'Bob' 'Carol' 'Dave']
Heights: [1.65 1.8  1.7  1.75]
Active members: [('Alice', 30, 1.65,  True) ('Carol', 35, 1.7 ,  True)
 ('Dave', 28, 1.75,  True)]


In [37]:
# --- Real use case: storing ML experiment results ---
results_dtype = np.dtype([
    ('model',     'U20'),
    ('fold',      np.int32),
    ('train_acc', np.float32),
    ('val_acc',   np.float32),
    ('time_s',    np.float32),
])

results = np.array([
    ('LogisticReg',  1, 0.92, 0.88, 1.2),
    ('LogisticReg',  2, 0.91, 0.87, 1.1),
    ('RandomForest', 1, 0.98, 0.90, 5.3),
    ('RandomForest', 2, 0.97, 0.89, 5.1),
], dtype=results_dtype)

print('Mean val_acc per model:')
for model in np.unique(results['model']):
    mask = results['model'] == model
    mean_val = results['val_acc'][mask].mean()
    print(f'  {model}: {mean_val:.4f}')

Mean val_acc per model:
  LogisticReg: 0.8750
  RandomForest: 0.8950


> ### INTERVIEW QUESTION
> **Q: When would you use a NumPy structured array instead of a Pandas DataFrame?**
>
> **A:** Structured arrays are preferable when: (1) you need minimal dependencies and lower memory overhead, (2) you're working in a NumPy-centric pipeline and don't need Pandas' alignment/groupby/merge features, (3) you're interfacing with C code via ctypes or Cython. Pandas DataFrames are backed by NumPy arrays internally but add significant overhead per-column. For simple record storage with vectorized field access, structured arrays are faster and leaner.
>
> ### WHY THIS MATTERS for ML/DS
> Structured arrays appear in HDF5 file reading (h5py), legacy scientific datasets, and when building lightweight result-tracking systems without the Pandas dependency.

---
## 8. Linear Algebra Deep Dive -- `np.linalg`

Core operations that underpin nearly all ML algorithms.

In [38]:
# --- Eigendecomposition: eig and eigh ---
# eig: general matrices (may return complex eigenvalues)
# eigh: symmetric/Hermitian matrices (guarantees real eigenvalues, faster)

# Covariance matrix is always symmetric -> use eigh
X = np.random.randn(100, 5)
cov = np.cov(X.T)   # shape (5,5)

eigenvalues, eigenvectors = np.linalg.eigh(cov)
print('Eigenvalues (sorted ascending):', eigenvalues)
print('Eigenvectors shape:', eigenvectors.shape)   # each COLUMN is an eigenvector

# Verify: A v = lambda v for each pair
for i in range(5):
    Av = cov @ eigenvectors[:, i]
    lv = eigenvalues[i] * eigenvectors[:, i]
    print(f'  eigenvector {i} check: {np.allclose(Av, lv)}')

Eigenvalues (sorted ascending): [0.62150755 0.79655475 0.96099488 1.22814381 1.43409198]
Eigenvectors shape: (5, 5)
  eigenvector 0 check: True
  eigenvector 1 check: True
  eigenvector 2 check: True
  eigenvector 3 check: True
  eigenvector 4 check: True


In [39]:
# --- SVD: Singular Value Decomposition ---
# A = U @ diag(s) @ Vt
# U: left singular vectors
# s: singular values (always non-negative, sorted descending)
# Vt: right singular vectors transposed

A = np.random.randn(4, 6)
U, s, Vt = np.linalg.svd(A, full_matrices=False)  # economy SVD

print('A shape:', A.shape)
print('U shape:', U.shape)
print('s shape:', s.shape)
print('Vt shape:', Vt.shape)

# Reconstruct A
A_reconstructed = U @ np.diag(s) @ Vt
print('Reconstruction error:', np.linalg.norm(A - A_reconstructed))

# Low-rank approximation (truncated SVD -- like PCA)
k = 2
A_lowrank = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
print(f'Low-rank (k={k}) approximation error: {np.linalg.norm(A - A_lowrank):.4f}')

A shape: (4, 6)
U shape: (4, 4)
s shape: (4,)
Vt shape: (4, 6)
Reconstruction error: 3.0244982157138154e-15
Low-rank (k=2) approximation error: 1.8758


In [40]:
# --- Matrix inverse and solving linear systems ---
A = np.array([[3., 1.], [1., 2.]])
b = np.array([9., 8.])

# Method 1: explicit inverse (AVOID for numerical stability)
x_inv = np.linalg.inv(A) @ b
print('Via inv:', x_inv)

# Method 2: np.linalg.solve (preferred -- uses LU decomposition)
x_solve = np.linalg.solve(A, b)
print('Via solve:', x_solve)

# Verify
print('Residual:', np.linalg.norm(A @ x_solve - b))

Via inv: [2. 3.]
Via solve: [2. 3.]
Residual: 0.0


In [41]:
# --- Norms ---
v = np.array([3., 4.])
M = np.array([[1., 2.], [3., 4.]])

print('Vector L2 norm:', np.linalg.norm(v))               # 5.0
print('Vector L1 norm:', np.linalg.norm(v, ord=1))         # 7.0
print('Vector Linf norm:', np.linalg.norm(v, ord=np.inf))  # 4.0

print('Matrix Frobenius norm:', np.linalg.norm(M))
print('Matrix spectral norm:', np.linalg.norm(M, ord=2))   # largest singular value

Vector L2 norm: 5.0
Vector L1 norm: 7.0
Vector Linf norm: 4.0
Matrix Frobenius norm: 5.477225575051661
Matrix spectral norm: 5.464985704219043


In [42]:
# --- Determinant and rank ---
A = np.array([[1., 2.], [3., 4.]])
print('Determinant:', np.linalg.det(A))   # -2.0

# Rank
B = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]])   # rank-deficient
print('Rank of B:', np.linalg.matrix_rank(B))   # 2, not 3

# slogdet: numerically stable log of absolute determinant
sign, logdet = np.linalg.slogdet(A)
print('slogdet:', sign, logdet, '-> det =', sign * np.exp(logdet))

Determinant: -2.0000000000000004
Rank of B: 2
slogdet: -1.0 0.6931471805599455 -> det = -2.0000000000000004


In [43]:
# --- Pseudo-inverse (Moore-Penrose) ---
# Used in least-squares regression: beta = pinv(X) @ y
# Works even when X is not square or is rank-deficient

X = np.random.randn(100, 5)
y = np.random.randn(100)

# Least squares via pinv
beta_pinv = np.linalg.pinv(X) @ y

# Preferred: lstsq (more numerically stable)
beta_lstsq, residuals, rank, sv = np.linalg.lstsq(X, y, rcond=None)
print('Solutions match:', np.allclose(beta_pinv, beta_lstsq))

Solutions match: True


> ### INTERVIEW QUESTION
> **Q: When would you use `np.linalg.solve` instead of `np.linalg.inv`? What is the difference between `eig` and `eigh`?**
>
> **A:** `solve(A, b)` is preferred over `inv(A) @ b` because it uses LU decomposition -- more numerically stable (avoids explicitly computing the inverse) and roughly 3x faster. Only use `inv` when you need the inverse matrix itself for multiple right-hand sides.  
> `eig` works on any square matrix and may return complex eigenvalues. `eigh` is optimized for symmetric/Hermitian matrices, guarantees real eigenvalues sorted ascending, and is faster. Always use `eigh` for covariance matrices and kernel matrices.
>
> ### WHY THIS MATTERS for ML/DS
> PCA uses SVD or eigendecomposition of the covariance matrix. Linear regression uses least-squares. Mahalanobis distance requires matrix inverse. Understanding which linalg tool to use affects both correctness and performance.

---
## 9. Random Number Generation -- New API vs Legacy

NumPy 1.17+ introduced a new `Generator` API (`np.random.default_rng`). It's faster, statistically superior, and the recommended approach.

### Legacy (avoid in new code):
```python
np.random.seed(42)
np.random.randn(10)
```

### New API (use this):
```python
rng = np.random.default_rng(42)
rng.standard_normal(10)
```

In [44]:
# --- New API: default_rng ---
rng = np.random.default_rng(seed=42)

samples = rng.standard_normal(size=(3, 4))
print('Standard normal:\n', samples)

u = rng.uniform(0, 1, size=5)
print('Uniform:', u)

ints = rng.integers(0, 10, size=10)
print('Integers:', ints)

Standard normal:
 [[ 0.30471708 -1.03998411  0.7504512   0.94056472]
 [-1.95103519 -1.30217951  0.1278404  -0.31624259]
 [-0.01680116 -0.85304393  0.87939797  0.77779194]]
Uniform: [0.64386512 0.82276161 0.4434142  0.22723872 0.55458479]
Integers: [8 0 8 8 2 6 1 7 7 3]


In [45]:
# --- Reproducibility: seeding ---
rng1 = np.random.default_rng(seed=123)
rng2 = np.random.default_rng(seed=123)

a = rng1.standard_normal(5)
b = rng2.standard_normal(5)
print('Same seed, same results:', np.allclose(a, b))

rng3 = np.random.default_rng(seed=456)
c = rng3.standard_normal(5)
print('Different seed, different results:', not np.allclose(a, c))

Same seed, same results: True
Different seed, different results: True


In [46]:
# --- Spawning independent RNGs for parallel work ---
parent_rng = np.random.default_rng(seed=42)

child_rngs = parent_rng.spawn(4)
for i, child in enumerate(child_rngs):
    print(f'Worker {i}: {child.standard_normal(3)}')

Worker 0: [0.41832997 0.60557617 0.02878786]
Worker 1: [ 1.25449437  0.60628944 -1.3401776 ]
Worker 2: [-1.64939902  1.40599898  0.72347118]
Worker 3: [-0.10042535  1.4601151  -1.58161994]


In [47]:
# --- Common distributions in ML ---
rng = np.random.default_rng(0)

# Xavier/Glorot initialization
fan_in, fan_out = 512, 256
limit = np.sqrt(6 / (fan_in + fan_out))
W = rng.uniform(-limit, limit, size=(fan_in, fan_out))
print('Xavier weights -- std:', round(float(W.std()), 4),
      'expected:', round(float(2*limit / np.sqrt(12)), 4))

# He initialization for ReLU
std = np.sqrt(2 / fan_in)
W_he = rng.standard_normal((fan_in, fan_out)) * std
print('He init -- std:', round(float(W_he.std()), 4), 'expected:', round(float(std), 4))

# Dropout mask
keep_prob = 0.8
mask = rng.random(size=(100,)) < keep_prob
print('Dropout mask -- keep rate:', round(float(mask.mean()), 3))

Xavier weights -- std: 0.051 expected: 0.051
He init -- std: 0.0625 expected: 0.0625
Dropout mask -- keep rate: 0.8


In [48]:
# --- Legacy API comparison (do NOT mix with new API) ---
np.random.seed(42)
legacy_sample = np.random.randn(5)
print('Legacy:', legacy_sample)

rng = np.random.default_rng(42)
new_sample = rng.standard_normal(5)
print('New API:', new_sample)
# Note: same seed does NOT give same values between legacy and new API
# They use different underlying generators (MT19937 vs PCG64)

Legacy: [ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337]
New API: [ 0.30471708 -1.03998411  0.7504512   0.94056472 -1.95103519]


> ### INTERVIEW QUESTION
> **Q: What is the advantage of `np.random.default_rng()` over `np.random.seed()`?**
>
> **A:** The new `Generator` API (default_rng) is: (1) **faster** -- uses PCG64 bit generator vs MT19937, (2) **statistically better** -- passes more randomness tests, (3) **thread-safe** -- each generator object has independent state (no global state mutation), (4) **spawnable** -- `rng.spawn(N)` creates N independent sub-generators for parallel work. Legacy `np.random.seed()` mutates global state, causing subtle bugs in multi-threaded code. Always use `default_rng` in new code.
>
> ### WHY THIS MATTERS for ML/DS
> Reproducibility is critical for experiment tracking. Thread-safety matters when using `joblib.Parallel` or multiprocessing for cross-validation.

---
## 10. Performance Tricks

### Key principles:
1. Pre-allocate arrays with `np.empty` (not `np.zeros`) when you'll overwrite all values
2. Use in-place operations (`+=`, `*=`) to avoid temporary arrays
3. Avoid unnecessary copies -- understand views
4. Use appropriate dtypes (float32 vs float64)
5. `np.empty` vs `np.zeros` vs `np.ones`

In [49]:
# --- Pre-allocation vs append ---
N = 100_000

# Bad: growing list then converting
t0 = time.perf_counter()
lst = []
for i in range(N):
    lst.append(i ** 2)
arr = np.array(lst)
t_append = time.perf_counter() - t0

# Good: pre-allocate
t0 = time.perf_counter()
arr2 = np.empty(N, dtype=np.int64)
for i in range(N):
    arr2[i] = i ** 2
t_prealloc = time.perf_counter() - t0

# Best: fully vectorized
t0 = time.perf_counter()
arr3 = np.arange(N) ** 2
t_vec = time.perf_counter() - t0

print(f'Append+convert: {t_append*1000:.2f} ms')
print(f'Pre-allocate:   {t_prealloc*1000:.2f} ms')
print(f'Vectorized:     {t_vec*1000:.3f} ms')
print(f'Vectorized speedup over append: {t_append/t_vec:.0f}x')

Append+convert: 23.55 ms
Pre-allocate:   22.37 ms
Vectorized:     0.355 ms
Vectorized speedup over append: 66x


In [50]:
# --- np.empty vs np.zeros ---
size = (1000, 1000)

t0 = time.perf_counter()
for _ in range(100):
    a = np.empty(size)
t_empty = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(100):
    b = np.zeros(size)
t_zeros = time.perf_counter() - t0

print(f'np.empty: {t_empty*1000:.2f} ms')
print(f'np.zeros: {t_zeros*1000:.2f} ms')
print(f'zeros is {t_zeros/t_empty:.1f}x slower (must write zeros to memory)')
print()
print('Use np.empty when you WILL overwrite all values immediately')
print('Use np.zeros when you need zero initialization')

np.empty: 0.26 ms
np.zeros: 18.31 ms
zeros is 69.6x slower (must write zeros to memory)

Use np.empty when you WILL overwrite all values immediately
Use np.zeros when you need zero initialization


In [51]:
# --- In-place operations avoid temporary arrays ---
a = np.random.randn(1_000_000)
b = np.random.randn(1_000_000)

# Out-of-place: allocates temporary for a+b, then multiplies
t0 = time.perf_counter()
for _ in range(10):
    c = (a + b) * 2.0
t_outplace = time.perf_counter() - t0

# In-place: no extra allocation
c = a.copy()
t0 = time.perf_counter()
for _ in range(10):
    c[:] = a
    c += b
    c *= 2.0
t_inplace = time.perf_counter() - t0

print(f'Out-of-place: {t_outplace*1000:.2f} ms')
print(f'In-place:     {t_inplace*1000:.2f} ms')
print(f'Speedup: {t_outplace/t_inplace:.2f}x')

Out-of-place: 17.92 ms
In-place:     19.89 ms
Speedup: 0.90x


In [52]:
# --- dtype choice: float32 vs float64 ---
N = 1_000_000

a64 = np.random.randn(N).astype(np.float64)
a32 = a64.astype(np.float32)

print(f'float64 size: {a64.nbytes / 1024:.1f} KB')
print(f'float32 size: {a32.nbytes / 1024:.1f} KB  (2x smaller)')

t0 = time.perf_counter()
for _ in range(100): _ = a64.sum()
t64 = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(100): _ = a32.sum()
t32 = time.perf_counter() - t0

print(f'float64 sum: {t64*1000:.3f} ms')
print(f'float32 sum: {t32*1000:.3f} ms')

float64 size: 7812.5 KB
float32 size: 3906.2 KB  (2x smaller)


float64 sum: 37.321 ms
float32 sum: 32.036 ms


In [53]:
# --- Memory usage of arrays ---
arrays = {
    'bool':    np.ones(1000, dtype=bool),
    'int8':    np.ones(1000, dtype=np.int8),
    'int32':   np.ones(1000, dtype=np.int32),
    'int64':   np.ones(1000, dtype=np.int64),
    'float32': np.ones(1000, dtype=np.float32),
    'float64': np.ones(1000, dtype=np.float64),
    'complex': np.ones(1000, dtype=complex),
}

print('Memory usage for 1000 elements:')
for name, arr in arrays.items():
    print(f'  {name:10s}: {arr.nbytes:6d} bytes  ({arr.itemsize} bytes/element)')

Memory usage for 1000 elements:
  bool      :   1000 bytes  (1 bytes/element)
  int8      :   1000 bytes  (1 bytes/element)
  int32     :   4000 bytes  (4 bytes/element)
  int64     :   8000 bytes  (8 bytes/element)
  float32   :   4000 bytes  (4 bytes/element)
  float64   :   8000 bytes  (8 bytes/element)
  complex   :  16000 bytes  (16 bytes/element)


> ### INTERVIEW QUESTION
> **Q: What is the difference between `np.empty`, `np.zeros`, and `np.ones`? When does it matter?**
>
> **A:** All three allocate a new array, but differ in initialization: `np.zeros` fills with 0 (writes to all memory), `np.ones` fills with 1, `np.empty` leaves memory uninitialized. `np.empty` is faster when you're about to overwrite all elements. For safety, use `np.zeros` as the default. The performance difference becomes significant for large arrays or when allocating inside tight loops.
>
> ### WHY THIS MATTERS for ML/DS
> Model training pre-allocates gradient buffers. Feature engineering may create many temporary arrays. Understanding allocation costs helps profile and optimize pipelines.

---
## 11. `np.apply_along_axis` vs Vectorized

`np.apply_along_axis` applies a 1D function along a specified axis. It looks convenient but is rarely the best choice for performance.

In [54]:
# --- apply_along_axis example ---
M = np.random.randn(1000, 10)

def normalize_row(row):
    return (row - row.mean()) / (row.std() + 1e-8)

# Using apply_along_axis
t0 = time.perf_counter()
for _ in range(10):
    result_apply = np.apply_along_axis(normalize_row, axis=1, arr=M)
t_apply = time.perf_counter() - t0

# Fully vectorized equivalent
t0 = time.perf_counter()
for _ in range(10):
    means = M.mean(axis=1, keepdims=True)
    stds  = M.std(axis=1, keepdims=True) + 1e-8
    result_vec = (M - means) / stds
t_vec = time.perf_counter() - t0

print(f'apply_along_axis: {t_apply*1000:.2f} ms')
print(f'Vectorized:       {t_vec*1000:.2f} ms')
print(f'Vectorized speedup: {t_apply/t_vec:.1f}x')
print(f'Results match: {np.allclose(result_apply, result_vec)}')

apply_along_axis: 290.72 ms
Vectorized:       2.28 ms
Vectorized speedup: 127.7x
Results match: True


In [55]:
# --- When apply_along_axis IS useful ---
# When the per-1D-slice function cannot be vectorized easily

def top_k_indices(row, k=3):
    return np.argsort(row)[-k:]

M = np.random.randn(5, 10)
top_indices = np.apply_along_axis(top_k_indices, axis=1, arr=M)
print('Top 3 indices per row:\n', top_indices)
print('Shape:', top_indices.shape)  # (5, 3)

# Vectorized equivalent for this specific case:
top_indices_vec = np.argsort(M, axis=1)[:, -3:]
print('Match:', np.array_equal(top_indices, top_indices_vec))

Top 3 indices per row:
 [[4 2 9]
 [1 9 7]
 [5 8 3]
 [1 6 0]
 [6 7 9]]
Shape: (5, 3)
Match: True


In [56]:
# --- apply_over_axes: different from apply_along_axis ---
M = np.arange(24).reshape(2, 3, 4)
result = np.apply_over_axes(np.sum, M, [0, 2])
print('Original shape:', M.shape)
print('After apply_over_axes sum [0,2] shape:', result.shape)
print(result)

Original shape: (2, 3, 4)
After apply_over_axes sum [0,2] shape: (1, 3, 1)
[[[ 60]
  [ 92]
  [124]]]


> ### INTERVIEW QUESTION
> **Q: When should you use `np.apply_along_axis` vs writing a vectorized solution?**
>
> **A:** Prefer vectorized solutions whenever possible -- they run in C and can be 10-100x faster. `apply_along_axis` calls your Python function once per slice (still O(N) Python calls). Use it only when the per-slice function is too complex to vectorize, uses external non-NumPy functions, or code clarity matters more than performance for a non-bottleneck. Always benchmark with realistic data sizes.
>
> ### WHY THIS MATTERS for ML/DS
> Row-wise normalization, per-sample statistics, and custom aggregations appear constantly in preprocessing. The vectorized patterns using `keepdims=True` are essential to know.

---
## 12. Masked Arrays -- Handling Missing Data in NumPy

`np.ma` provides arrays with a boolean mask indicating which values are invalid or missing. Operations automatically ignore masked values.

In [57]:
import numpy.ma as ma

# --- Create masked arrays ---
data = np.array([1., 2., -999., 4., 5., -999., 7.])
mask = data == -999.

masked = ma.array(data, mask=mask)
print('Masked array:', masked)
print('Data:', masked.data)
print('Mask:', masked.mask)
print('Filled (replace masked with 0):', masked.filled(0))

Masked array: [1.0 2.0 -- 4.0 5.0 -- 7.0]
Data: [   1.    2. -999.    4.    5. -999.    7.]
Mask: [False False  True False False  True False]
Filled (replace masked with 0): [1. 2. 0. 4. 5. 0. 7.]


In [58]:
# --- Operations ignore masked values ---
data = np.array([1., 2., np.nan, 4., 5., np.nan, 7.])
masked = ma.masked_invalid(data)   # auto-mask NaN and Inf

print('Mean (ignoring NaN):', masked.mean())
print('Std  (ignoring NaN):', masked.std())
print('Sum  (ignoring NaN):', masked.sum())

# Compare with regular numpy
print('Regular mean (propagates NaN):', data.mean())
print('nanmean:', np.nanmean(data))

Mean (ignoring NaN): 3.8
Std  (ignoring NaN): 2.1354156504062622
Sum  (ignoring NaN): 19.0
Regular mean (propagates NaN): nan
nanmean: 3.8


In [59]:
# --- Masked array from condition ---
sensor_data = np.array([23.1, 24.5, -40.0, 22.8, 150.0, 23.9, 24.1])

# Mask values outside physical range
masked = ma.masked_outside(sensor_data, -50, 100)  # mask values outside valid sensor range
print('Valid readings:', masked.compressed())   # removes masked values
print('Mean valid reading:', masked.mean())

# Mask specific value
masked2 = ma.masked_equal(sensor_data, -40.0)
print('After masking -40:', masked2)

Valid readings: [ 23.1  24.5 -40.   22.8  23.9  24.1]
Mean valid reading: 13.066666666666668
After masking -40: [23.1 24.5 -- 22.8 150.0 23.9 24.1]


In [60]:
# --- 2D masked array with imputation ---
matrix = np.array([[1., 2., 3.],
                   [4., np.nan, 6.],
                   [7., 8., np.nan]])

masked_matrix = ma.masked_invalid(matrix)

print('Column means (ignoring NaN):', masked_matrix.mean(axis=0))
print('Row means (ignoring NaN):',    masked_matrix.mean(axis=1))

# Fill with column means for imputation
col_means = masked_matrix.mean(axis=0)
filled = masked_matrix.filled(col_means)
print('\nImputed matrix:\n', filled)

Column means (ignoring NaN): [4.0 5.0 4.5]
Row means (ignoring NaN): [2.0 5.0 7.5]

Imputed matrix:
 [[1.  2.  3. ]
 [4.  5.  6. ]
 [7.  8.  4.5]]


> ### INTERVIEW QUESTION
> **Q: What is `numpy.ma` and when would you use it over `np.nan`?**
>
> **A:** `numpy.ma` provides masked arrays where specific values are logically excluded from computations via a boolean mask. Use it over `np.nan` when: (1) your dtype is integer (NaN only works with floats), (2) you need to track which values are missing separately from the data values, (3) you want automatic propagation of the invalid status through operations. `np.nan` is simpler but limited to float arrays; masked arrays work with any dtype.
>
> ### WHY THIS MATTERS for ML/DS
> Before loading data into Pandas, NumPy masked arrays provide a lightweight missing-value handling mechanism. They're also used in geospatial/satellite data where invalid pixels must be excluded.

---
## 13. Common Interview Traps

These are the gotchas that trip up even experienced NumPy users.

In [61]:
# --- TRAP 1: shape (n,) vs (n,1) vs (1,n) ---
a = np.array([1, 2, 3])       # shape (3,)  -- 1D
b = np.array([[1, 2, 3]])     # shape (1,3) -- 2D row vector
c = np.array([[1],[2],[3]])   # shape (3,1) -- 2D column vector

print('a shape:', a.shape)
print('b shape:', b.shape)
print('c shape:', c.shape)

print('\na @ a:', a @ a)      # scalar dot product: 14
print('b @ c:', b @ c)        # (1,1) matrix

# Safe reshape
a_col = a.reshape(-1, 1)   # or a[:, np.newaxis]
a_row = a.reshape(1, -1)   # or a[np.newaxis, :]
print('\na_col shape:', a_col.shape)
print('a_row shape:', a_row.shape)

a shape: (3,)
b shape: (1, 3)
c shape: (3, 1)

a @ a: 14
b @ c: [[14]]

a_col shape: (3, 1)
a_row shape: (1, 3)


In [62]:
# --- TRAP 2: 0-d arrays (scalars wrapped in array) ---
a = np.array(42)       # 0-d array
print('Shape:', a.shape)    # ()
print('ndim:', a.ndim)      # 0

b = np.array([42])     # 1-d array with one element
print('\nb.shape:', b.shape)    # (1,)
print('a == b:', a == b)         # True (value equal) but different shapes!

# Convert 0-d to Python scalar
scalar = a.item()
print('a.item():', scalar, type(scalar))

# len() fails on 0-d
try:
    len(a)
except TypeError as e:
    print('len(0-d array) error:', e)

Shape: ()
ndim: 0

b.shape: (1,)
a == b: [ True]
a.item(): 42 <class 'int'>
len(0-d array) error: len() of unsized object


In [63]:
# --- TRAP 3: Integer overflow ---
# NumPy integers have fixed bit-width -- no Python auto-promotion!
a = np.array([127], dtype=np.int8)
print('int8 max:', np.iinfo(np.int8).max)   # 127
print('127 + 1 =', (a + np.int8(1))[0])    # -128 !! overflow wraps

# Python int: no overflow
print('Python: 127 + 1 =', 127 + 1)   # 128

# Safe: use larger dtype
b = np.array([127], dtype=np.int64)
print('int64: 127 + 1 =', (b + 1)[0])   # 128

# Check dtype limits
for dt in [np.int8, np.int16, np.int32, np.int64]:
    info = np.iinfo(dt)
    print(f'  {dt.__name__:6s}: min={info.min:22d}  max={info.max}')

int8 max: 127
127 + 1 = -128
Python: 127 + 1 = 128
int64: 127 + 1 = 128
  int8  : min=                  -128  max=127
  int16 : min=                -32768  max=32767
  int32 : min=           -2147483648  max=2147483647
  int64 : min=  -9223372036854775808  max=9223372036854775807


In [64]:
# --- TRAP 4: Float precision ---
print('0.1 + 0.2 == 0.3:', 0.1 + 0.2 == 0.3)   # False!
print('0.1 + 0.2:', 0.1 + 0.2)

# Use np.isclose / np.allclose for float comparison
a = np.array([0.1 + 0.2])
b = np.array([0.3])
print('np.isclose:', np.isclose(a, b))   # True

# float32 has less precision than float64
x64 = np.float64(1e-7) + np.float64(1.0)
x32 = np.float32(1e-7) + np.float32(1.0)
print('\nfloat64 result:', x64 - 1.0)
print('float32 result:', x32 - 1.0)  # might be 0.0 (precision lost!)

0.1 + 0.2 == 0.3: False
0.1 + 0.2: 0.30000000000000004
np.isclose: [ True]

float64 result: 1.0000000005838672e-07
float32 result: 1.1920929e-07


In [65]:
# --- TRAP 5: Chained indexing assignment ---
M = np.zeros((3, 3))

# BUG: fancy indexing returns a copy, assignment is lost
row_indices = [0, 1]
M[row_indices][0] = 99   # This does NOTHING to M
print('After M[[0,1]][0] = 99 (chained):\n', M)   # still zeros!

# Correct:
M[row_indices[0]] = 99
print('After M[0] = 99:\n', M)

After M[[0,1]][0] = 99 (chained):
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
After M[0] = 99:
 [[99. 99. 99.]
 [ 0.  0.  0.]
 [ 0.  0.  0.]]


In [66]:
# --- TRAP 6: Boolean array in if statement ---
a = np.array([True, False, True])

print('np.all(a):', np.all(a))   # False
print('np.any(a):', np.any(a))   # True

# WRONG: 'if a:' raises ValueError for arrays with >1 element
try:
    if a:
        pass
except ValueError as e:
    print('Error:', e)

# Correct:
if np.all(a):
    print('All true')
else:
    print('Not all true -- use np.all() for aggregate truth check')

np.all(a): False
np.any(a): True
Error: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
Not all true -- use np.all() for aggregate truth check


In [67]:
# --- TRAP 7: Python sum vs np.sum ---
a = np.array([1e10, -1e10, 1.0])

# Python's sum: left-to-right (can lose precision)
print('Python sum:', sum(a))   # might lose 1.0

# NumPy's sum: pairwise summation (better precision)
print('NumPy sum:', np.sum(a))

big = np.random.randn(1_000_000)
t0 = time.perf_counter()
_ = sum(big)
t_py = time.perf_counter() - t0

t0 = time.perf_counter()
_ = np.sum(big)
t_np = time.perf_counter() - t0

print(f'\nPython sum: {t_py*1000:.1f} ms')
print(f'NumPy sum:  {t_np*1000:.2f} ms')
print(f'NumPy is {t_py/t_np:.0f}x faster')

Python sum: 1.0
NumPy sum: 1.0



Python sum: 101.9 ms
NumPy sum:  0.59 ms
NumPy is 173x faster


In [68]:
# --- TRAP 8: Broadcasting shape trick (interview classic) ---
a = np.array([1, 2, 3])           # shape (3,)
b = np.array([4, 5, 6]).reshape(3, 1)  # shape (3,1)

result = a + b
print('a shape:', a.shape)
print('b shape:', b.shape)
print('result shape:', result.shape)  # (3,3) !! NOT (3,)
print('result:\n', result)
print()
print('This is element [i,j] = a[j] + b[i]')
print('-- an outer-product-like addition --')

a shape: (3,)
b shape: (3, 1)
result shape: (3, 3)
result:
 [[5 6 7]
 [6 7 8]
 [7 8 9]]

This is element [i,j] = a[j] + b[i]
-- an outer-product-like addition --


---
## Quick Reference: What to Remember for Interviews

| Topic | Key Point |
|-------|-----------|
| Views vs Copies | Slicing -> view; fancy/boolean indexing -> copy; check `.base` |
| Broadcasting | Pad left with 1s; size-1 dims stretch; incompatible sizes error |
| Memory layout | C-order: row-contiguous; strides = bytes per step; `ascontiguousarray` |
| Fancy indexing | Returns copy; chained assignment via copy is a silent bug |
| Vectorization | `np.vectorize` is NOT fast; use `np.where`/`np.select` instead |
| einsum | `'ij,jk->ik'` = matmul; unindexed dims are summed; batch ops with extra dim |
| linalg | `solve` > `inv`; `eigh` for symmetric; `lstsq` for overdetermined systems |
| RNG | Use `default_rng(seed)` not `np.random.seed()`; thread-safe |
| Performance | `np.empty` > `np.zeros` when overwriting; in-place `+=` saves memory |
| Shapes | `(n,)` != `(n,1)` != `(1,n)`; use `reshape(-1,1)` to convert |
| Overflow | NumPy integers wrap at bit boundary; float32 loses precision vs float64 |
| Comparison | Never `==` on floats; use `np.isclose`; never `if array:` use `np.all` |

---
*End of Advanced NumPy Interview Preparation Notebook*